In [2]:
import pandas as pd

matches = pd.read_csv('data/matches.csv')
deliveries = pd.read_csv('data/deliveries.csv')

print('matches shape:', matches.shape)
print('deliveries shape:', deliveries.shape)

matches shape: (1095, 20)
deliveries shape: (260920, 17)


In [3]:
print("MATCHES columns:")
print(matches.columns.tolist())

print("\nDELIVERIES columns:")
print(deliveries.columns.tolist())

MATCHES columns:
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

DELIVERIES columns:
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']


In [4]:
print("MATCHES nulls:")
print(matches.isnull().sum())

print("\nDELIVERIES nulls:")
print(deliveries.isnull().sum())

MATCHES nulls:
id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64

DELIVERIES nulls:
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64


In [5]:
print("Matches date range:")
print(matches['date'].min(), "to", matches['date'].max())

print("\nSeasons:")
print(sorted(matches['season'].unique()))

print("\nUnique teams:")
print(sorted(matches['team1'].unique()))

Matches date range:
2008-04-18 to 2024-05-26

Seasons:
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024']

Unique teams:
['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Delhi Daredevils', 'Gujarat Lions', 'Gujarat Titans', 'Kings XI Punjab', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Rising Pune Supergiants', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


In [6]:
# Check how many wins each team name has
print(matches['winner'].value_counts())


winner
Mumbai Indians                 144
Chennai Super Kings            138
Kolkata Knight Riders          131
Royal Challengers Bangalore    116
Rajasthan Royals               112
Kings XI Punjab                 88
Sunrisers Hyderabad             88
Delhi Daredevils                67
Delhi Capitals                  48
Deccan Chargers                 29
Gujarat Titans                  28
Punjab Kings                    24
Lucknow Super Giants            24
Gujarat Lions                   13
Pune Warriors                   12
Rising Pune Supergiant          10
Royal Challengers Bengaluru      7
Kochi Tuskers Kerala             6
Rising Pune Supergiants          5
Name: count, dtype: int64


In [7]:
team_name_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Kings XI Punjab': 'Punjab Kings'
}

team_columns = ['team1', 'team2', 'toss_winner', 'winner']

for col in team_columns:
    matches[col] = matches[col].replace(team_name_mapping)
    
print(matches['winner'].value_counts())

winner
Mumbai Indians                 144
Chennai Super Kings            138
Kolkata Knight Riders          131
Royal Challengers Bengaluru    123
Sunrisers Hyderabad            117
Delhi Capitals                 115
Rajasthan Royals               112
Punjab Kings                   112
Gujarat Titans                  28
Lucknow Super Giants            24
Rising Pune Supergiants         15
Gujarat Lions                   13
Pune Warriors                   12
Kochi Tuskers Kerala             6
Name: count, dtype: int64


In [8]:
# Season normalisation — extract the start year from each season value

def normalize_season(season):
    # season is a string like '2007/08' or '2021'
    # splitting on '/' gives either ['2007', '08'] or ['2021']
    # taking [0] always gives the first part — the start year
    return int(season.split('/')[0])

matches['season'] = matches['season'].apply(normalize_season)

# Verify
print(matches['season'].unique())

[2007 2009 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022
 2023 2024]


In [9]:
# Fill missing city values with 'Unknown'
matches['city'] = matches['city'].fillna('Unknown')

# Verify no nulls remain
print(matches['city'].isnull().sum())

0


In [10]:
# Fill structural nulls in deliveries with 'none'
columns_to_fill = ['extras_type', 'player_dismissed', 'dismissal_kind', 'fielder']

for col in columns_to_fill:
    deliveries[col] = deliveries[col].fillna('none')

# Verify no nulls remain in these columns
print(deliveries[columns_to_fill].isnull().sum())

extras_type         0
player_dismissed    0
dismissal_kind      0
fielder             0
dtype: int64


In [11]:
matches.to_csv('data/matches_cleaned.csv', index=False)
deliveries.to_csv('data/deliveries_cleaned.csv', index=False)

print("Cleaned files saved successfully")

Cleaned files saved successfully


In [12]:
all_players = pd.concat([
    deliveries['batter'],
    deliveries['bowler'],
    deliveries['non_striker'],
    deliveries['fielder']
]).unique()

In [13]:
all_players = [p for p in all_players if p != 'none']

In [14]:
dim_player = pd.DataFrame({
    'player_id': range(1, len(all_players) + 1),
    'player_name': sorted(all_players)
})

print(dim_player.shape)
dim_player.head()

(741, 2)


,player_id,player_name
0,1,A Ashish Reddy
1,2,A Badoni
2,3,A Chandila
3,4,A Chopra
4,5,A Choudhary


In [15]:
all_teams = pd.concat([
    deliveries['batting_team'],
    deliveries['bowling_team']
]).unique()

dim_team = pd.DataFrame({
    'team_id': range(1, len(all_teams) + 1),
    'team_name': sorted(all_teams)
})

print(dim_team.shape)
dim_team.head()

(19, 2)


,team_id,team_name
0,1,Chennai Super Kings
1,2,Deccan Chargers
2,3,Delhi Capitals
3,4,Delhi Daredevils
4,5,Gujarat Lions


In [16]:
deliveries_team_columns = ['batting_team', 'bowling_team']

for col in deliveries_team_columns:
    deliveries[col] = deliveries[col].replace(team_name_mapping)

# Rebuild dim_team now that deliveries is fixed
all_teams = pd.concat([
    deliveries['batting_team'],
    deliveries['bowling_team']
]).unique()

dim_team = pd.DataFrame({
    'team_id': range(1, len(all_teams) + 1),
    'team_name': sorted(all_teams)
})

print(dim_team.shape)
dim_team.head()

(14, 2)


,team_id,team_name
0,1,Chennai Super Kings
1,2,Delhi Capitals
2,3,Gujarat Lions
3,4,Gujarat Titans
4,5,Kochi Tuskers Kerala


In [17]:
dim_match = matches[['id', 'season', 'date', 'city', 'venue', 'team1', 'team2',
                       'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin']].copy()

dim_match = dim_match.rename(columns={'id': 'match_id'})

print(dim_match.shape)
dim_match.head()

(1095, 12)


,match_id,season,date,city,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin
0,335982,2007,2008-04-18,Bangalore,M Chinnaswamy Stadium,Royal Challengers Bengaluru,Kolkata Knight Riders,Royal Challengers Bengaluru,field,Kolkata Knight Riders,runs,140.0
1,335983,2007,2008-04-19,Chandigarh,"Punjab Cricket Association Stadium, Mohali",Punjab Kings,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0
2,335984,2007,2008-04-19,Delhi,Feroz Shah Kotla,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,bat,Delhi Capitals,wickets,9.0
3,335985,2007,2008-04-20,Mumbai,Wankhede Stadium,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,Royal Challengers Bengaluru,wickets,5.0
4,335986,2007,2008-04-20,Kolkata,Eden Gardens,Kolkata Knight Riders,Sunrisers Hyderabad,Sunrisers Hyderabad,bat,Kolkata Knight Riders,wickets,5.0


In [18]:
# Step 1: Create a player name → player_id lookup dictionary
player_lookup = dict(zip(dim_player['player_name'], dim_player['player_id']))

# Step 2: Create a team name → team_id lookup dictionary
team_lookup = dict(zip(dim_team['team_name'], dim_team['team_id']))

# Step 3: Build fact_deliveries by mapping names to IDs
fact_deliveries = deliveries.copy()

fact_deliveries['batter_id'] = fact_deliveries['batter'].map(player_lookup)
fact_deliveries['bowler_id'] = fact_deliveries['bowler'].map(player_lookup)
fact_deliveries['non_striker_id'] = fact_deliveries['non_striker'].map(player_lookup)
fact_deliveries['fielder_id'] = fact_deliveries['fielder'].map(player_lookup)

fact_deliveries['batting_team_id'] = fact_deliveries['batting_team'].map(team_lookup)
fact_deliveries['bowling_team_id'] = fact_deliveries['bowling_team'].map(team_lookup)

print(fact_deliveries.shape)
fact_deliveries[['batter', 'batter_id', 'bowler', 'bowler_id']].head()


(260920, 23)


,batter,batter_id,bowler,bowler_id
0,SC Ganguly,590,P Kumar,464
1,BB McCullum,111,P Kumar,464
2,BB McCullum,111,P Kumar,464
3,BB McCullum,111,P Kumar,464
4,BB McCullum,111,P Kumar,464


In [19]:
id_columns = ['batter_id', 'bowler_id', 'non_striker_id', 'fielder_id', 'batting_team_id', 'bowling_team_id']
print(fact_deliveries[id_columns].isnull().sum())

batter_id               0
bowler_id               0
non_striker_id          0
fielder_id         251566
batting_team_id         0
bowling_team_id         0
dtype: int64


In [20]:
fact_deliveries['fielder_id'] = fact_deliveries['fielder_id'].fillna(0).astype(int)

# Verify
print(fact_deliveries['fielder_id'].isnull().sum())
print(fact_deliveries['fielder_id'].dtype)

0
int64


In [21]:
fact_deliveries['delivery_id'] = range(1, len(fact_deliveries) + 1)

# Keep only the columns that belong in a clean fact table
fact_deliveries_final = fact_deliveries[[
    'delivery_id', 'match_id', 'inning', 'over', 'ball',
    'batting_team_id', 'bowling_team_id',
    'batter_id', 'bowler_id', 'non_striker_id', 'fielder_id',
    'batsman_runs', 'extra_runs', 'total_runs',
    'extras_type', 'is_wicket', 'dismissal_kind'
]].copy()

print(fact_deliveries_final.shape)
fact_deliveries_final.head()

(260920, 17)


,delivery_id,match_id,inning,over,ball,batting_team_id,bowling_team_id,batter_id,bowler_id,non_striker_id,fielder_id,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,dismissal_kind
0,1,335982,1,0,1,6,13,590,464,111,0,0,1,1,legbyes,0,none
1,2,335982,1,0,2,6,13,111,464,590,0,0,0,0,none,0,none
2,3,335982,1,0,3,6,13,111,464,590,0,0,1,1,wides,0,none
3,4,335982,1,0,4,6,13,111,464,590,0,0,0,0,none,0,none
4,5,335982,1,0,5,6,13,111,464,590,0,0,0,0,none,0,none


In [22]:
import sqlite3

# Create a connection to a new SQLite database file
conn = sqlite3.connect('ipl_pipeline.db')

# Load all four tables into the database
dim_player.to_sql('dim_player', conn, if_exists='replace', index=False)
dim_team.to_sql('dim_team', conn, if_exists='replace', index=False)
dim_match.to_sql('dim_match', conn, if_exists='replace', index=False)
fact_deliveries_final.to_sql('fact_deliveries', conn, if_exists='replace', index=False)

print("All tables loaded successfully")
conn.close()

All tables loaded successfully


In [23]:
conn = sqlite3.connect('ipl_pipeline.db')

# Check all tables exist in the database
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
print(tables)

# Check row counts for each table
for table in ['dim_player', 'dim_team', 'dim_match', 'fact_deliveries']:
    count = pd.read_sql(f"SELECT COUNT(*) as row_count FROM {table}", conn)
    print(f"{table}: {count['row_count'][0]} rows")

conn.close()

Tables in database:
              name
0       dim_player
1         dim_team
2        dim_match
3  fact_deliveries
dim_player: 741 rows
dim_team: 14 rows
dim_match: 1095 rows
fact_deliveries: 260920 rows


In [24]:
conn = sqlite3.connect('ipl_pipeline.db')

# Query 1: Most successful teams by total wins
query1 = """
SELECT 
    dm.winner,
    COUNT(*) as total_wins
FROM dim_match dm
WHERE dm.winner IS NOT NULL
    AND dm.winner != ''
GROUP BY dm.winner
ORDER BY total_wins DESC
LIMIT 10
"""

result1 = pd.read_sql(query1, conn)
print("Top 10 teams by wins:")
print(result1)

Top 10 teams by wins:
                        winner  total_wins
0               Mumbai Indians         144
1          Chennai Super Kings         138
2        Kolkata Knight Riders         131
3  Royal Challengers Bengaluru         123
4          Sunrisers Hyderabad         117
5               Delhi Capitals         115
6             Rajasthan Royals         112
7                 Punjab Kings         112
8               Gujarat Titans          28
9         Lucknow Super Giants          24


In [25]:
# Query 2: Top 10 run scorers of all time
query2 = """
SELECT 
    dp.player_name,
    SUM(fd.batsman_runs) as total_runs
FROM fact_deliveries fd
JOIN dim_player dp ON fd.batter_id = dp.player_id
GROUP BY dp.player_name
ORDER BY total_runs DESC
LIMIT 10
"""

result2 = pd.read_sql(query2, conn)
print("Top 10 run scorers:")
print(result2)

Top 10 run scorers:
      player_name  total_runs
0         V Kohli        8014
1        S Dhawan        6769
2       RG Sharma        6630
3       DA Warner        6567
4        SK Raina        5536
5        MS Dhoni        5243
6  AB de Villiers        5181
7        CH Gayle        4997
8      RV Uthappa        4954
9      KD Karthik        4843


In [26]:
# Query 3: Top 10 wicket takers of all time
query3 = """
SELECT 
    dp.player_name,
    SUM(fd.is_wicket) as total_wickets
FROM fact_deliveries fd
JOIN dim_player dp ON fd.bowler_id = dp.player_id
GROUP BY dp.player_name
ORDER BY total_wickets DESC
LIMIT 10
"""

result3 = pd.read_sql(query3, conn)
print("Top 10 wicket takers:")
print(result3)

Top 10 wicket takers:
  player_name  total_wickets
0   YS Chahal            213
1    DJ Bravo            207
2   PP Chawla            201
3   SP Narine            200
4    R Ashwin            198
5     B Kumar            195
6  SL Malinga            188
7    A Mishra            183
8   JJ Bumrah            182
9   RA Jadeja            169


In [27]:
# Query 4: Total runs scored per season
query4 = """
SELECT 
    dm.season,
    SUM(fd.total_runs) as total_runs,
    COUNT(DISTINCT fd.match_id) as matches_played
FROM fact_deliveries fd
JOIN dim_match dm ON fd.match_id = dm.match_id
GROUP BY dm.season
ORDER BY dm.season ASC
"""

result4 = pd.read_sql(query4, conn)
print("Runs per season:")
print(result4)

Runs per season:
    season  total_runs  matches_played
0     2007       17937              58
1     2009       35236             117
2     2011       21154              73
3     2012       22453              74
4     2013       22602              76
5     2014       18931              60
6     2015       18353              59
7     2016       18862              60
8     2017       18786              59
9     2018       19901              60
10    2019       19434              60
11    2020       19416              60
12    2021       18637              60
13    2022       24395              74
14    2023       25688              74
15    2024       25971              71


In [28]:
# Query 5: Most economical bowlers (min 50 overs bowled)
query5 = """
SELECT 
    dp.player_name,
    SUM(fd.total_runs) as runs_conceded,
    COUNT(*) as balls_bowled,
    ROUND(COUNT(*) / 6.0, 1) as overs_bowled,
    ROUND(SUM(fd.total_runs) * 6.0 / COUNT(*), 2) as economy_rate
FROM fact_deliveries fd
JOIN dim_player dp ON fd.bowler_id = dp.player_id
GROUP BY dp.player_name
HAVING overs_bowled >= 50
ORDER BY economy_rate ASC
LIMIT 10
"""

result5 = pd.read_sql(query5, conn)
print("Most economical bowlers (min 50 overs):")
print(result5)

Most economical bowlers (min 50 overs):
        player_name  runs_conceded  balls_bowled  overs_bowled  economy_rate
0          A Kumble           1089           983         163.8          6.65
1        GD McGrath            366           329          54.8          6.67
2    M Muralitharan           1765          1581         263.5          6.70
3           J Yadav            447           398          66.3          6.74
4         SP Narine           4672          4146         691.0          6.76
5          DW Steyn           2583          2282         380.3          6.79
6  RE van der Merwe            515           455          75.8          6.79
7        DL Vettori            894           785         130.8          6.83
8       Rashid Khan           3340          2901         483.5          6.91
9           J Botha            818           709         118.2          6.92


In [29]:
conn.close()